# 02 — Model comparison and open-world evaluation

Reproducible plots for the final four pipelines. The compact CSV summaries are tracked in `results/`, so this notebook runs immediately after cloning.

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
sns.set_theme(style="whitegrid", context="notebook")
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RESULTS = ROOT / "results"
models = pd.read_csv(RESULTS / "model_comparison.csv")
flags = pd.read_csv(RESULTS / "flagging_summary.csv")
early = pd.read_csv(RESULTS / "early_classification.csv")
models

## Closed-set classification and held-out unknown separation

In [ ]:
metrics = models.melt("model", value_vars=["accuracy", "balanced_accuracy", "macro_f1", "unknown_auroc"], var_name="metric", value_name="score")
plt.figure(figsize=(12, 5.2))
ax = sns.barplot(data=metrics, x="model", y="score", hue="metric", palette="deep")
ax.set(ylim=(0, 1), title="Final model comparison", xlabel="Pipeline", ylabel="Score")
ax.legend(title="Metric", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout(); plt.show()

## Classification–novelty trade-off

The upper-right region is desirable. The ensemble leads classification, while DriftMamba has the strongest unknown AUROC.

In [ ]:
plt.figure(figsize=(8, 5.5))
ax = sns.scatterplot(data=models, x="macro_f1", y="unknown_auroc", hue="model", size="accuracy", sizes=(150, 420), palette="deep")
for row in models.itertuples(): ax.annotate(row.model, (row.macro_f1, row.unknown_auroc), xytext=(6, 5), textcoords="offset points")
ax.set(title="Classification versus unknown separation", xlim=(.60, .75), ylim=(.53, .73))
ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left"); plt.tight_layout(); plt.show()

## Flag composition and operational selectivity

Known flagged flows are false rejections or rare legitimate behavior; unknown flagged flows are successful held-out detections.

In [ ]:
plot_flags = flags.set_index("model")[["known_flagged", "unknown_flagged"]]
ax = plot_flags.plot.bar(stacked=True, figsize=(11, 5), color=["#EF4444", "#0F9D8A"])
ax.set(title="Composition of open-world flags", xlabel="Pipeline", ylabel="Flows flagged (of 2,000)")
ax.legend(["Known flows rejected", "Held-out unknown detected"]); plt.xticks(rotation=0); plt.tight_layout(); plt.show()

In [ ]:
rates = flags.melt("model", value_vars=["unknown_recall", "known_acceptance"], var_name="rate", value_name="score")
plt.figure(figsize=(10, 4.8)); ax = sns.barplot(data=rates, x="model", y="score", hue="rate", palette=["#F59E0B", "#2563EB"])
ax.set(ylim=(0, 1), title="Unknown recall versus known-traffic acceptance", xlabel="Pipeline", ylabel="Rate")
plt.xticks(rotation=0); plt.tight_layout(); plt.show()

## Early-flow performance

This curve shows the value gained as more packets become available to DriftMamba.

In [ ]:
early_long = early.melt("observed_packets", var_name="metric", value_name="score")
plt.figure(figsize=(9, 4.8)); ax = sns.lineplot(data=early_long, x="observed_packets", y="score", hue="metric", marker="o", linewidth=2.5)
ax.set(ylim=(0, 1), xticks=early["observed_packets"], title="Accuracy–latency trade-off", xlabel="Observed packets", ylabel="Score")
plt.tight_layout(); plt.show()

## Compact comparison table

In [ ]:
models.style.format({"accuracy":"{:.3f}", "balanced_accuracy":"{:.3f}", "macro_f1":"{:.3f}", "unknown_auroc":"{:.3f}"}).background_gradient(subset=["accuracy", "balanced_accuracy", "macro_f1", "unknown_auroc"], cmap="YlGnBu")